In [ ]:
import sys,os,re
import numpy             as np
import matplotlib.pyplot as plt
import pandas            as pd
import seaborn           as sb

from source_code.galdist import galaxy_distribution

from itertools import product
from copy      import deepcopy
from time      import time

from scipy.interpolate import interp1d
from scipy.integrate   import trapz

import warnings
warnings.filterwarnings('ignore')

import matplotlib
from matplotlib import rc
from matplotlib.pyplot import cm
from matplotlib.colors import LogNorm

rc('text', usetex=True)
rc('font', family='serif')
matplotlib.rcParams.update({'font.size': 18})

red    = '#8e001c'
yellow = '#ffb302'

sidelegend = {'bbox_to_anchor': (1.04,0.5), 
              'loc': "center left",
              'frameon': False}

#sys.path.insert(0,camb_path)
import camb
print(camb.__path__)

# Settings

In [ ]:
#Euclid survey specifications, N_gw in [10^5-10^6] according to ET
galaxy_specs = {'fsky': 0.35, 
                'gal_per_arcmin': 30.,
                'sigma_eps': 0.3,  #sigma associated to noise for GC and WL
                'Nbin_ell': 20,
                'lmin': 10,
                'lmax': 1500}

GW_specs = {'fsky': 0.35, 
            'N_gw': 10**5, 
            'sigma_eps_gw': 0.005} #sigma associated to noise (d_L) for GW-WL

analysis_settings = {'Nbin_ell': 20,
                     'lmin': 10,
                     'lmax': 1500}

use_obs  = ['GC','WL','GWWL']



## Preliminary calculations

Setting up some quantities that will be used later

In [ ]:
lmin = np.log10(analysis_settings['lmin'])
lmax = np.log10(analysis_settings['lmax'])
N    = analysis_settings['Nbin_ell']

ell_lims = np.logspace(lmin,lmax,N) #creation of array-> N bin log spaced
ells     = np.array([int(ell) for ell in 0.5*(ell_lims[:-1]+ell_lims[1:])]) 
#evaluation of middle points of each bin 
deltas   = (ell_lims[1:]-ell_lims[:-1]) #evaluation of the amplitude of each bin

# Fiducial cosmology and CAMB settings

In [ ]:
#Cosmological parameters describing the LCDM Universe
fiducial = {'ombh2': 0.022445,
            'omch2': 0.1205579307,
            'ns': 0.96,
            'As': 2.12605e-09,
            'tau': 0.05,
            'H0': 67.,
            'w': -1.,
            'wa': 0.,
            'mnu': 0.06,
            'a0': - 0.007589,
            'a1' :  0.002008,
            'a2' : - 0.004127,
            'a3' :  0.002918,
            'a4' : -0.0006784,
            #'A_IA': 1.72,
            #'eta_IA': -0.41,
            'b0_poly': 0.830703,
            'b1_poly': 1.190547,
            'b2_poly': -0.928357,
            'b3_poly': 0.423292}
MG_params={'MG_flag': 0}
fiducial.update(MG_params)

#camb_path = '/Users/chiaradeleo/myenv/lib/python3.12/site-packages'


# Galaxy distributions (external)

These cells create the galaxy distribution object that is needed by the obs computation.
We save all this info in a dictionary that will 

In [ ]:
distributions = {}

In [ ]:
if 'GC' in use_obs:
    dist = galaxy_distribution(survey='Euclid-10')
    bin_lims = dist.galdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_gc = len(bin_lims)-1
    
    distributions['GC'] = {'dist': dist.galdict['binned_dist'],
                           'Nbins': Nbins_gc,
                           'zmean': bin_mids}
    
if 'WL' in use_obs:
    dist = galaxy_distribution(survey='Euclid-10')
    bin_lims = dist.galdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_wl = len(bin_lims)-1
    
    distributions['WL'] = {'dist': dist.galdict['binned_dist'],
                           'Nbins': Nbins_wl,
                           'zmean': bin_mids}
    
sys.path.insert(0,'/Users/chiaradeleo/Desktop/6x2pt_IA_EFT_switch/source_code_old/')
if 'GWC' in use_obs:
    from gwdist import gw_distribution
    
    gwdist  = gw_distribution()
    
    
    ngwbin = (GW_specs['N_gw']/(len(gwdist.z_bins)-1))
    Nbins_gw = len(gwdist.z_bins)-1
    
    bin_lims = gwdist.gwdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_wl = len(bin_lims)-1
    
    distributions['GWC'] = {'dist': gwdist.ni_gw,
                           'Nbins': Nbins_gw,
                           'zmean': bin_mids}
if 'GWWL' in use_obs:
    from gwdist import gw_distribution
    
    gwdist  = gw_distribution()
    
    
    ngwbin = (GW_specs['N_gw']/(len(gwdist.z_bins)-1))
    Nbins_gw = len(gwdist.z_bins)-1
    
    bin_lims = gwdist.gwdict['bin_lims']
    bin_mids = 0.5*(bin_lims[:-1]+bin_lims[1:])
    Nbins_wl = len(bin_lims)-1
    
    distributions['GWWL'] = {'dist': gwdist.ni_gw,
                           'Nbins': Nbins_gw,
                           'zmean': bin_mids}

In [ ]:
data_root = './mock_data/'


if MG_params: 
    if MG_params['MG_flag']==0:
        data_root = data_root+'LCDM_test_'
    else:
        data_root = data_root+f'MGflag_{eft_params["MG_flag"]}_test_'
        
if 'GC' in use_obs and 'WL' in use_obs:
    data_root = data_root+'gal'
elif 'GC' in use_obs:
    data_root = data_root+'GC'
elif 'WL' in use_obs:
    data_root = data_root+'WL'
if 'GWWL' in use_obs and 'GWC' in use_obs:
        data_root = data_root+'GWs'
elif 'GWC' in use_obs:
        data_root = data_root+'GWC'
elif 'GWWL' in use_obs:
        data_root = data_root+'GWWL'
print(data_root)


In [ ]:
maxbins=0
if 'GC' in use_obs:
    Nbins_gc=distributions['GC']['Nbins']
    maxbins=max(Nbins_gc, maxbins)
if 'WL' in use_obs:
    Nbins_wl=distributions['WL']['Nbins']
    maxbins=max(Nbins_wl, maxbins)
if 'GWWL' in use_obs:
    Nbins_gwl=distributions['GWWL']['Nbins']
    maxbins=max(Nbins_gwl, maxbins)
if 'GWC' in use_obs:
    Nbins_gwc=distributions['GWC']['Nbins']
    maxbins=max(Nbins_gwc, maxbins)

# Observables computation

In [ ]:
from source_code.compute_obs_sources import get_obs
extra={}

In [ ]:
fiducial_MG = {'ombh2': 0.022445,
            'omch2': 0.1205579307,
            'ns': 0.96,
            'As': 2.12605e-09,
            'tau': 0.05,
            'H0': 67.,
            'w': -1.,
            'wa': 0.,
            'mnu': 0.06,
            'a0': - 0.007589,
            'a1' :  0.002008,
            'a2' : - 0.004127,
            'a3' :  0.002918,
            'a4' : -0.0006784,
            'omegab': 0.05,
            'sigma8': 0.84,
            'omegam' : 0.31,
            'b0_poly': 0.830703,
            'b1_poly': 1.190547,
            'b2_poly': -0.928357,
            'b3_poly': 0.423292}
MG_params={'MG_flag': 1,
           'pure_MG_flag': 2,
           'musigma_par': 1,
           'DE_model': 0,
           'sigma0': -0.17,
           'mu0': -0.58}
fiducial_MG.update(MG_params)

In [ ]:
settings={'camb_path': camb.__path__,
         'case': 'simple',
         'calculation': 'CAMB',
          'extra':extra}
calc_obs = get_obs(fiducial,distributions,ells,settings,feedback=True)


# Constructing covariance

**WARNING:** 
- currently setup for galaxies only
- assumes Gaussian covariance

In [ ]:
from source_code.covariance_utils import covariance_einsum

Nell = {k: [0.]*len(ells) for k in calc_obs.Cls.columns}


for i in range(1,maxbins+1):
    if 'GC' in use_obs and i<=Nbins_gc:
        ngalbin = (galaxy_specs['gal_per_arcmin']/Nbins_gc)*3600*(180/np.pi)**2
        Nell['G{}xG{}'.format(i,i)] = [(1/ngalbin)]*len(ells)
    if 'WL' in use_obs and i<= Nbins_wl:
        ngalbin = (galaxy_specs['gal_per_arcmin']/Nbins_wl)*3600*(180/np.pi)**2
        Nell['L{}xL{}'.format(i,i)] = [(galaxy_specs['sigma_eps']**2/(2*ngalbin))]*len(ells)
    if 'GWWL' in use_obs and i<= Nbins_gwl:
        ngwlbin = GW_specs['N_gw']/Nbins_gwl 
        Nell['WL{}xWL{}'.format(i,i)] += [(GW_specs['sigma_eps_gw']**2/ngwlbin)]*len(ells)
    if 'GWC' in use_obs  and i<= Nbins_gwc:
        ngwcbin = GW_specs['N_gw']/Nbins_gwc 
        Nell['WC{}xWC{}'.format(i,i)] += [(GW_specs['sigma_eps_gw']**2/ngwcbin)]*len(ells)

fsky      = galaxy_specs['fsky']
Delta_ell = deltas
j=0
obs_list = []
if 'GC' in use_obs:
    obs_list.append('G')
    j+=1
if 'WL' in use_obs:
    obs_list.append('L')
    j+=1
if 'GWWL' in use_obs:
    obs_list.append('WL')
    j+=1
if 'GWC' in use_obs:
    obs_list.append('WC')
    j+=1



err_for_cov = np.zeros((j,j,len(ells),Nbins_wl,Nbins_wl))
cls_for_cov = np.zeros((j,j,len(ells),Nbins_wl,Nbins_wl))

if 'GC' in use_obs and 'WL' in use_obs and 'GWWL' in use_obs and 'GWC' in use_obs:
    for o1,obs1 in enumerate(obs_list):
        if o1==0:
            Nbins1=Nbins_gc
        elif o1==1:
            Nbins1=Nbins_wl
        elif o1==2:
            Nbins1=Nbins_gwl
        elif o1==3:
            Nbins1=Nbins_gwc
        for o2,obs2 in enumerate(obs_list):
            if o2==0:
                Nbins2=Nbins_gc
            elif o2==1:
                Nbins2=Nbins_wl
            elif o2==2:
                Nbins2=Nbins_gwl
            elif o2==3:
                Nbins2=Nbins_gwc
            for i in range(Nbins1):
                for j in range(Nbins2):
                    for ell_ind,ell in enumerate(ells):
                        if obs1 == obs2 and j<i:
                            Nell[obs1+str(i+1)+'x'+obs2+str(j+1)] = Nell[obs1+str(j+1)+'x'+obs2+str(i+1)]
                            calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)] = calc_obs.Cls[obs1+str(j+1)+'x'+obs2+str(i+1)]
                        
    
                        err_for_cov[o1,o2,ell_ind,i,j] = Nell[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]
                        cls_for_cov[o1,o2,ell_ind,i,j] = calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]


elif 'GC' in use_obs and 'WL' in use_obs and 'GWWL' in use_obs:
    for o1,obs1 in enumerate(obs_list):
        if o1==0:
            Nbins1=Nbins_gc
        elif o1==1:
            Nbins1=Nbins_wl
        elif o1==2:
            Nbins1=Nbins_gwl
        for o2,obs2 in enumerate(obs_list):
            if o2==0:
                Nbins2=Nbins_gc
            elif o2==1:
                Nbins2=Nbins_wl
            elif o2==2:
                Nbins2=Nbins_gwl
            for i in range(Nbins1):
                for j in range(Nbins2):
                    for ell_ind,ell in enumerate(ells):
                        if obs1 == obs2 and j<i:
                            Nell[obs1+str(i+1)+'x'+obs2+str(j+1)] = Nell[obs1+str(j+1)+'x'+obs2+str(i+1)]
                            calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)] = calc_obs.Cls[obs1+str(j+1)+'x'+obs2+str(i+1)]
                        
    
                        err_for_cov[o1,o2,ell_ind,i,j] = Nell[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]
                        cls_for_cov[o1,o2,ell_ind,i,j] = calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]

elif 'GC' in use_obs and 'WL' in use_obs and 'GWC' in use_obs:
    for o1,obs1 in enumerate(obs_list):
        if o1==0:
            Nbins1=Nbins_gc
        elif o1==1:
            Nbins1=Nbins_wl
        elif o1==2:
            Nbins1=Nbins_gwc
        for o2,obs2 in enumerate(obs_list):
            if o2==0:
                Nbins2=Nbins_gc
            elif o2==1:
                Nbins2=Nbins_wl
            elif o2==2:
                Nbins2=Nbins_gwc
            for i in range(Nbins1):
                for j in range(Nbins2):
                    for ell_ind,ell in enumerate(ells):
                        if obs1 == obs2 and j<i:
                            Nell[obs1+str(i+1)+'x'+obs2+str(j+1)] = Nell[obs1+str(j+1)+'x'+obs2+str(i+1)]
                            calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)] = calc_obs.Cls[obs1+str(j+1)+'x'+obs2+str(i+1)]
                        
    
                        err_for_cov[o1,o2,ell_ind,i,j] = Nell[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]
                        cls_for_cov[o1,o2,ell_ind,i,j] = calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]


elif 'GWWL' in use_obs and 'GWC' in use_obs:
    for o1,obs1 in enumerate(obs_list):
        if o1==0:
            Nbins1=Nbins_gwl
        elif o1==1:
            Nbins1=Nbins_gwc
        for o2,obs2 in enumerate(obs_list):
            if o2==0:
                Nbins2=Nbins_gwl
            elif o2==1:
                Nbins2=Nbins_gwc
            for i in range(Nbins1):
                for j in range(Nbins2):
                    for ell_ind,ell in enumerate(ells):
                        if obs1 == obs2 and j<i:
                            Nell[obs1+str(i+1)+'x'+obs2+str(j+1)] = Nell[obs1+str(j+1)+'x'+obs2+str(i+1)]
                            calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)] = calc_obs.Cls[obs1+str(j+1)+'x'+obs2+str(i+1)]
                        
    
                        err_for_cov[o1,o2,ell_ind,i,j] = Nell[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]
                        cls_for_cov[o1,o2,ell_ind,i,j] = calc_obs.Cls[obs1+str(i+1)+'x'+obs2+str(j+1)][ell_ind]




covmat_diag = covariance_einsum(cls_for_cov,err_for_cov,fsky,ells,Delta_ell,return_only_diagonal_ells=True)
#print(covmat_diag)


# Creating dataset

## Packing covmat

In [ ]:
def split_num(s):
    head = s.rstrip('0123456789')
    tail = s[len(head):]
    return head, tail

In [ ]:
if 'WL' in use_obs: 
    WLcols  = ['L{}xL{}'.format(i,j) for i in range(1,Nbins_wl+1) for j in range(i,Nbins_wl+1)]
if 'GC' in use_obs: 
    GCcols  = ['G{}xG{}'.format(i,j) for i in range(1,Nbins_gc+1) for j in range(i,Nbins_gc+1)]    
if 'GWWL' in use_obs: 
    GWWLcols  = ['WL{}xWL{}'.format(i,j) for i in range(1,Nbins_gwl+1) for j in range(i,Nbins_gwl+1)]
if 'GWC' in use_obs: 
    GWCcols  = ['WC{}xWC{}'.format(i,j) for i in range(1,Nbins_gwc+1) for j in range(i,Nbins_gwc+1)]
if 'GC' in  use_obs and 'WL' in use_obs: 
    GGLcols = ['G{}xL{}'.format(i,j) for i in range(1,Nbins_gc+1) for j in range(1,Nbins_wl+1)]
if 'WL' in use_obs and 'GWC' in use_obs:
    LGWCcols = ['L{}xWC{}'.format(i,j) for i in range(1,Nbins_wl+1) for j in range(1,Nbins_gwc+1)]
if 'GC' in  use_obs and 'GWC' in  use_obs:
    GGWCcols = ['G{}xWC{}'.format(i,j) for i in range(1,Nbins_gc+1) for j in range(1,Nbins_gwc+1)]
if 'GC' in  use_obs and 'GWWL' in  use_obs:
    GGWLcols = ['G{}xWL{}'.format(i,j) for i in range(1,Nbins_gc+1) for j in range(1,Nbins_gwl+1)]
if 'WL' in use_obs and 'GWWL' in use_obs:
    LGWLcols = ['L{}xWL{}'.format(i,j) for i in range(1,Nbins_wl+1) for j in range(1,Nbins_gwl+1)]
if 'GWC' in use_obs and 'GWWL' in use_obs:
    GWCGWLcols = ['WC{}xWL{}'.format(i,j) for i in range(1,Nbins_gwc+1) for j in range(1,Nbins_gwl+1)]

In [ ]:
all_cols=[]
if 'WL' in use_obs: 
    all_cols = all_cols+WLcols
if 'GC' in use_obs:
    if 'WL' in use_obs:
        all_cols = all_cols+GGLcols+GCcols
    else:
        all_cols = all_cols+GCcols
if 'GWC' in use_obs:
    all_cols = all_cols+GWCcols
    if 'GC' in use_obs:
        all_cols = all_cols+GGWCcols
    if 'WL' in use_obs:
        all_cols = all_cols+LGWCcols
if 'GWWL' in use_obs:
    all_cols = all_cols + GWWLcols
    if 'GC' in use_obs:
        all_cols = all_cols+GGWLcols
    if 'WL' in use_obs:
        all_cols = all_cols+LGWLcols
    if 'GWC' in use_obs:
        all_cols = all_cols+GWCGWLcols


Ndof = len(all_cols)*len(ells)

In [ ]:
def str_to_ind(in_obs):
    if 'GC' in use_obs and 'WL' in use_obs and 'GWWL' in use_obs and 'GWC' in use_obs:
        if in_obs == 'G':
            ind = 0
        elif in_obs == 'L':
            ind = 1
        elif in_obs == 'WL':
            ind = 2
        elif in_obs == 'WC':
            ind = 3
    elif 'GC' in use_obs and 'WL' in use_obs and 'GWWL' in use_obs:
        if in_obs == 'G':
            ind = 0
        elif in_obs == 'L':
            ind = 1
        elif in_obs == 'WL':
            ind = 2
    elif 'GC' in use_obs and 'WL' in use_obs and 'GWC' in use_obs:
        if in_obs == 'G':
            ind = 0
        elif in_obs == 'L':
            ind = 1
        elif in_obs == 'WC':
            ind = 2
    elif 'GC' in use_obs and 'GWWL' in use_obs and 'GWC' in use_obs:
        if in_obs == 'G':
            ind = 0
        elif in_obs == 'WL':
            ind = 1
        elif in_obs == 'WC':
            ind = 2
    elif 'WL' in use_obs and 'GWWL' in use_obs and 'GWC' in use_obs:
        if in_obs == 'L':
            ind = 0
        elif in_obs == 'WL':
            ind = 1
        elif in_obs == 'WC':
            ind = 2
    elif 'GC' in use_obs and 'WL' in use_obs:
        if in_obs == 'G':
            ind = 0
        elif in_obs == 'L':
            ind = 1
    elif 'GC' in use_obs and 'GWWL' in use_obs:
        if in_obs == 'G':
            ind = 0
        elif in_obs == 'WL':
            ind = 1 
    elif 'WL' in use_obs and 'GWWL' in use_obs:
        if in_obs == 'L':
            ind = 0
        elif in_obs == 'WL':
            ind = 1 
    elif 'GC' in use_obs and 'GWC' in use_obs:
        if in_obs == 'G':
            ind = 0
        elif in_obs == 'WC':
            ind = 1 
    elif 'WL' in use_obs and 'GWC' in use_obs:
        if in_obs == 'L':
            ind = 0
        elif in_obs == 'WC':
            ind = 1
    elif 'GWWL' in use_obs and 'GWC' in use_obs:
        if in_obs == 'WL':
            ind = 0
        elif in_obs == 'WC':
            ind = 1                    
    else:
        ind = 0

    return ind




covmat_dict = {}

for ellind,ell in enumerate(ells):
    
    packed_covmat = pd.DataFrame(columns=all_cols,index=all_cols,dtype='float')
    
    for ind1,col in enumerate(all_cols):
        bin1,bin2 = re.split('x',col)
        oi1,i1 = split_num(bin1)
        oj1,j1 = split_num(bin2)
    
        for ind2,row in enumerate(all_cols):
            bin1,bin2 = re.split('x',row)
            oi2,i2 = split_num(bin1)
            oj2,j2 = split_num(bin2)
            
            packed_covmat.at[row,col] = covmat_diag[str_to_ind(oi1),str_to_ind(oj1),str_to_ind(oi2),str_to_ind(oj2),
                                                    ellind,int(i1)-1,int(j1)-1,int(i2)-1,int(j2)-1]
            
            packed_covmat.index = packed_covmat.columns
            
    covmat_dict[str(int(ell))] = packed_covmat

In [ ]:
plot_cov=False
if plot_cov==True:
    for ell in ells:
        plt.figure()
        plt.title(r'Covariance matrix at $\ell={}$'.format(int(ell)))
        sb.heatmap(covmat_dict[str(int(ell))],norm=LogNorm());

## Creating final Cls

In [ ]:
def get_realization(fiducial,covmats):

    noisy_cls = fiducial.copy()
    cols = [col for col in fiducial.columns if col != 'ells']

    for ind,ell in enumerate(fiducial['ells']):
        covmat = covmats[str(int(ell))]

        means  = [fiducial[col][ind] for col in cols]
        sample = np.random.multivariate_normal(means,covmat)
        
        for col_ind,col in enumerate(cols):
            noisy_cls.at[ind,col] = sample[col_ind]

    return noisy_cls

In [ ]:
noiseless_cls = pd.DataFrame(columns=['ells']+all_cols)

noiseless_cls['ells'] = [int(ell) for ell in ells]

for col in all_cols:
    noiseless_cls[col] = calc_obs.Cls[col]
    
noisy_cls = get_realization(noiseless_cls,covmat_dict)

## Some testing plots

In [ ]:
diag_error = noiseless_cls.copy()

for ind,ell in enumerate(ells):
    for col in all_cols:
        diag_error.at[ind,col] = np.sqrt(covmat_dict[str(int(ell))].at[col,col])

## Saving to file if requested

In [ ]:
if data_root != '':
    noiseless_cls.to_csv(data_root+'_Cls_noiseless.dat',sep='\t',header=True)
    noisy_cls.to_csv(data_root+'_Cls_noisy.dat',sep='\t',header=True)
    np.save(data_root+'_source_distribution.npy',distributions)
    np.save(data_root+'_covmat.npy',covmat_dict)